# TerraMind Base Flood Prediction Colab

workflow:
1. โหลด checkpoint TerraMind Base ที่ fine-tune จาก Sen1Floods11
2. อ่าน Sentinel-1 และ Sentinel-2 จาก Google Drive
3. แปลง Sentinel-1 เป็น dB และจัดรูปแบบเป็น VV/VH
4. จัด Sentinel-2 ให้เป็น 13 bands ตามที่โมเดลต้องการ
5. ทำ flood prediction แบบ tiled inference
6. export GeoTIFF, polygon, preview PNG และ summary CSV


# 0 — โฟลเดอร์ที่ควรเตรียมใน Google Drive

แนะนำให้เตรียมโฟลเดอร์นี้:

```text
MyDrive/Flood_Prediction_Package/
├── checkpoints/
│   └── best_terramind_base_sen1floods11.ckpt
├── input/
│   └── scenes/
│       ├── scene_01/
│       │   ├── S1.tif
│       │   └── S2.tif
│       ├── scene_02/
│       │   ├── S1.tif
│       │   └── S2.tif
│       └── ...
└── outputs/
```

## S1.tif ควรเป็นอะไร

ดีที่สุดคือ Sentinel-1 ที่มี 2 bands:

```text
Band 1 = VV
Band 2 = VH
```

แต่ notebook รองรับกรณีมีแค่ 1 band ด้วย โดยจะถือว่า band นั้นเป็น VV และสร้าง VH แบบประมาณเพื่อให้โมเดลรันได้

## S2.tif ควรเป็นอะไร

ดีที่สุดคือ Sentinel-2 L1C 13 bands

แต่ notebook รองรับ:
- 13 bands: ดีที่สุด
- 4 bands: RGB+NIR
- 3 bands: RGB

ถ้ามีแค่ RGB notebook จะเติม band ที่ขาดด้วยค่าเฉลี่ยจาก Sen1Floods11 เพื่อให้โมเดลรับ input ได้

In [ ]:
# ============================================================
# 1 — ติดตั้ง library
# ============================================================
# แนะนำให้เลือก GPU ก่อนรัน:
# Runtime → Change runtime type → GPU
# ============================================================

import sys
import subprocess

packages = [
    "terratorch[tortilla]==1.2.5",
    "setuptools<81",
    "rasterio",
    "geopandas",
    "shapely",
    "pyogrio",
    "scipy",
    "pandas",
    "matplotlib",
    "tqdm",
    "pyyaml",
]

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    *packages,
]

print("Running:")
print(" ".join(command))
subprocess.check_call(command)

print("ติดตั้งเสร็จแล้ว ถ้า cell ถัดไป import error ให้ Restart session แล้วรันต่อจาก cell 2")

In [ ]:
# ============================================================
# 2 — ตรวจสอบ environment และ GPU
# ============================================================

import sys
import warnings
import importlib.metadata

import numpy as np
import pandas as pd
import torch
import rasterio
import geopandas as gpd
import terratorch
import tacoreader

warnings.filterwarnings("ignore")

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"

print("=" * 100)
print("ENVIRONMENT CHECK")
print("=" * 100)
print("Python     :", sys.version.split()[0])
print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)
print("PyTorch    :", torch.__version__)
print("TerraTorch :", package_version("terratorch"))
print("Tacoreader :", package_version("tacoreader"))
print("Rasterio   :", rasterio.__version__)
print("GeoPandas  :", gpd.__version__)
print("CUDA       :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("ไม่พบ GPU: กรุณาเลือก Runtime → Change runtime type → GPU")

device = torch.device("cuda")
print("GPU        :", torch.cuda.get_device_name(0))
print("GPU memory :", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print("พร้อมใช้งาน")

In [ ]:
# ============================================================
# 3 — Mount Google Drive และตั้งค่าโฟลเดอร์
# ============================================================

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive")

# โฟลเดอร์ package สำหรับส่งต่อ
PACKAGE_ROOT = DRIVE_ROOT / "Flood_Prediction_Package"

# โฟลเดอร์ legacy จากงานทดลองเดิม
LEGACY_PROJECT_ROOT = DRIVE_ROOT / "TerraMind_Flood"

OUTPUT_ROOT = PACKAGE_ROOT / "outputs"
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

README_PATH = PACKAGE_ROOT / "README_วิธีเตรียมโฟลเดอร์.txt"
readme_lines = [
    "Flood Prediction Package",
    "",
    "โครงสร้างที่แนะนำ:",
    "",
    "Flood_Prediction_Package/",
    "├── checkpoints/",
    "│   └── best_terramind_base_sen1floods11.ckpt",
    "├── input/",
    "│   └── scenes/",
    "│       ├── scene_01/",
    "│       │   ├── S1.tif",
    "│       │   └── S2.tif",
    "│       ├── scene_02/",
    "│       │   ├── S1.tif",
    "│       │   └── S2.tif",
    "│       └── ...",
    "└── outputs/",
    "",
    "S1.tif: ดีที่สุดคือ 2 bands = VV, VH",
    "S2.tif: ดีที่สุดคือ Sentinel-2 L1C 13 bands",
    "Checkpoint: วาง best_terramind_base_sen1floods11.ckpt ใน checkpoints/",
]
README_PATH.write_text("\n".join(readme_lines), encoding="utf-8")

print("=" * 100)
print("PATH SETUP")
print("=" * 100)
print("DRIVE_ROOT          :", DRIVE_ROOT)
print("PACKAGE_ROOT        :", PACKAGE_ROOT)
print("LEGACY_PROJECT_ROOT :", LEGACY_PROJECT_ROOT)
print("OUTPUT_ROOT         :", OUTPUT_ROOT)
print("README              :", README_PATH)

In [ ]:
# ============================================================
# 4 — ตั้งค่า prediction
# ============================================================

import numpy as np

PATCH_SIZE = 512
STRIDE = 256
FLOOD_THRESHOLD = 0.55
MIN_COMPONENT_PIXELS = 30
MIN_POLYGON_AREA_M2 = 1000.0

# CRS สำหรับคำนวณพื้นที่ประเทศไทย (UTM zone 47N)
AREA_CRS = "EPSG:32647"

# ถ้า S2 ไม่มี CRS จะใช้ CRS ของ S1
USE_S1_CRS_WHEN_S2_CRS_MISSING = True

# Sentinel-1 DN มักเป็น scaled linear power
S1_SCALE_FACTOR = 10000.0

# ถ้า S1 มีแค่ 1 band: ใช้เป็น VV แล้วประมาณ VH ด้วย offset นี้
S1_VV_MEAN_DB = -12.599
S1_VH_MEAN_DB = -20.293
VH_OFFSET_DB = S1_VH_MEAN_DB - S1_VV_MEAN_DB

# ถ้า S2 เป็น RGB 3 bands ให้ถือว่าลำดับเป็น RGB
S2_RGB_ORDER = "RGB"

S2_MEAN = np.array(
    [2357.089, 2137.385, 2018.788, 2082.986, 2295.651, 2854.537, 3122.849,
     3040.560, 3306.481, 1473.847, 506.070, 2472.825, 1838.929],
    dtype=np.float32,
)

S2_STD = np.array(
    [1624.683, 1675.806, 1557.708, 1833.702, 1823.738, 1733.977, 1732.131,
     1679.732, 1727.260, 1024.687, 442.165, 1331.411, 1160.419],
    dtype=np.float32,
)

S1_MEAN = np.array([-12.599, -20.293], dtype=np.float32)
S1_STD = np.array([5.195, 5.890], dtype=np.float32)

print("=" * 100)
print("PREDICTION SETTINGS")
print("=" * 100)
print("PATCH_SIZE      :", PATCH_SIZE)
print("STRIDE          :", STRIDE)
print("FLOOD_THRESHOLD :", FLOOD_THRESHOLD)
print("AREA_CRS        :", AREA_CRS)
print("VH_OFFSET_DB    :", VH_OFFSET_DB)

In [ ]:
# ============================================================
# 5 — ค้นหา checkpoint อัตโนมัติ
# ============================================================

from pathlib import Path

search_roots = [
    PACKAGE_ROOT / "checkpoints",
    PACKAGE_ROOT / "checkpoint",
    LEGACY_PROJECT_ROOT / "checkpoints",
    LEGACY_PROJECT_ROOT,
]

checkpoint_candidates = []

for root in search_roots:
    if root.exists():
        checkpoint_candidates.extend(root.rglob("*.ckpt"))

checkpoint_candidates = sorted(
    set(checkpoint_candidates),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not checkpoint_candidates:
    raise FileNotFoundError(
        "ไม่พบ checkpoint .ckpt\n"
        "กรุณาวาง best_terramind_base_sen1floods11.ckpt ไว้ใน "
        f"{PACKAGE_ROOT / 'checkpoints'}"
    )

best_candidates = [p for p in checkpoint_candidates if "last" not in p.name.lower()]
CHECKPOINT = best_candidates[0] if best_candidates else checkpoint_candidates[0]

print("=" * 100)
print("CHECKPOINT FOUND")
print("=" * 100)

for p in checkpoint_candidates[:10]:
    print(f"{p.stat().st_size / 1024**2:8.2f} MB | {p}")

print("\nSelected checkpoint:")
print(CHECKPOINT)

In [ ]:
# ============================================================
# 6 — ค้นหา scene S1/S2 อัตโนมัติ
# ============================================================

import re
from pathlib import Path

def extract_scene_key(path):
    matches = re.findall(r"\d+", path.stem.lower())
    if matches:
        return str(int(matches[0])).zfill(2)
    return path.stem.lower()

def discover_scene_folder():
    scene_root = PACKAGE_ROOT / "input" / "scenes"
    if not scene_root.exists():
        return []

    pairs = []

    for scene_dir in sorted(scene_root.iterdir()):
        if not scene_dir.is_dir():
            continue

        s1 = scene_dir / "S1.tif"
        s2 = scene_dir / "S2.tif"

        if not s1.exists():
            s1_candidates = [p for p in scene_dir.rglob("*.tif") if "s1" in p.name.lower()]
            s1 = s1_candidates[0] if s1_candidates else None

        if not s2.exists():
            s2_candidates = [p for p in scene_dir.rglob("*.tif") if "s2" in p.name.lower()]
            s2 = s2_candidates[0] if s2_candidates else None

        if s1 and s2 and Path(s1).exists() and Path(s2).exists():
            pairs.append({"scene": scene_dir.name, "s1": Path(s1), "s2": Path(s2), "source": "package_scene_folder"})

    return pairs

def discover_s1_s2_folders():
    s1_dirs = [
        PACKAGE_ROOT / "input" / "S1",
        PACKAGE_ROOT / "input" / "s1",
        PACKAGE_ROOT / "S1",
        DRIVE_ROOT / "sat_images" / "crop-sat-img" / "S1-aoi",
    ]

    s2_dirs = [
        PACKAGE_ROOT / "input" / "S2",
        PACKAGE_ROOT / "input" / "s2",
        PACKAGE_ROOT / "S2",
        DRIVE_ROOT / "sat_images" / "crop-sat-img" / "S2-aoi",
    ]

    s1_dir = next((p for p in s1_dirs if p.exists()), None)
    s2_dir = next((p for p in s2_dirs if p.exists()), None)

    if s1_dir is None or s2_dir is None:
        return []

    s1_map = {extract_scene_key(p): p for p in sorted(s1_dir.glob("*.tif"))}
    s2_map = {extract_scene_key(p): p for p in sorted(s2_dir.glob("*.tif"))}

    pairs = []
    for key in sorted(set(s1_map) & set(s2_map)):
        pairs.append({"scene": key, "s1": s1_map[key], "s2": s2_map[key], "source": "paired_s1_s2_folders"})

    return pairs

def discover_single_legacy():
    s1 = DRIVE_ROOT / "sat_images" / "crop-sat-img" / "AOI_img" / "s1_vv_vh.tif"
    s2 = DRIVE_ROOT / "sat_images" / "crop-sat-img" / "AOI_img" / "s2_13band.tif"

    if s1.exists() and s2.exists():
        return [{"scene": "single", "s1": s1, "s2": s2, "source": "legacy_single_file"}]

    return []

scene_pairs = discover_scene_folder()

if not scene_pairs:
    scene_pairs = discover_s1_s2_folders()

if not scene_pairs:
    scene_pairs = discover_single_legacy()

if not scene_pairs:
    raise FileNotFoundError("ไม่พบคู่ S1/S2 กรุณาตรวจโฟลเดอร์ input")

print("=" * 100)
print("DISCOVERED SCENES")
print("=" * 100)

for pair in scene_pairs:
    print(f"Scene {pair['scene']} | {pair['source']}")
    print("  S1:", pair["s1"], "|", pair["s1"].exists())
    print("  S2:", pair["s2"], "|", pair["s2"].exists())

print("\nTotal scenes:", len(scene_pairs))

In [ ]:
# ============================================================
# 7 — ตรวจ metadata ของไฟล์ input
# ============================================================

import numpy as np
import pandas as pd
import rasterio
from IPython.display import display

def raster_report(path, scene, role):
    rows = []
    with rasterio.open(path) as src:
        for band_index in range(1, src.count + 1):
            band = src.read(band_index, masked=True)
            valid = band.compressed()
            rows.append(
                {
                    "scene": scene,
                    "role": role,
                    "file": Path(path).name,
                    "band": band_index,
                    "band_count": src.count,
                    "width": src.width,
                    "height": src.height,
                    "crs": str(src.crs),
                    "resolution_x": abs(src.transform.a),
                    "resolution_y": abs(src.transform.e),
                    "dtype": src.dtypes[band_index - 1],
                    "min": float(valid.min()) if valid.size else np.nan,
                    "max": float(valid.max()) if valid.size else np.nan,
                    "mean": float(valid.mean()) if valid.size else np.nan,
                    "median": float(np.median(valid)) if valid.size else np.nan,
                }
            )
    return rows

rows = []
for pair in scene_pairs:
    rows.extend(raster_report(pair["s1"], pair["scene"], "S1"))
    rows.extend(raster_report(pair["s2"], pair["scene"], "S2"))

report_df = pd.DataFrame(rows)
display(report_df)

print("ถ้า S1 มี 1 band หรือ S2 มี 3 bands ยังรันได้ แต่ให้ตีความเป็น prototype")

In [ ]:
# ============================================================
# 8 — โหลด TerraMind model
# ============================================================

import torch
from terratorch.tasks import SemanticSegmentationTask

MODEL_ARGS = {
    "backbone": "terramind_v1_base",
    "backbone_pretrained": False,
    "backbone_modalities": ["S2L1C", "S1GRD"],
    "backbone_merge_method": "mean",
    "necks": [
        {"name": "SelectIndices", "indices": [2, 5, 8, 11]},
        {"name": "ReshapeTokensToImage", "remove_cls_token": False},
        {"name": "LearnedInterpolateToPyramidal"},
    ],
    "decoder": "UNetDecoder",
    "decoder_channels": [512, 256, 128, 64],
    "head_dropout": 0.1,
    "num_classes": 2,
}

model = SemanticSegmentationTask.load_from_checkpoint(
    checkpoint_path=str(CHECKPOINT),
    model_factory="EncoderDecoderFactory",
    model_args=MODEL_ARGS,
    map_location=device,
)

model = model.to(device)
model.eval()

print("=" * 100)
print("MODEL LOADED")
print("=" * 100)
print("Checkpoint:", CHECKPOINT)
print("Device    :", device)
print("GPU       :", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 9 — ฟังก์ชันเตรียม S1 และ S2
# ============================================================

from pathlib import Path
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

def make_safe_profile(source_profile, width, height, count, dtype, nodata):
    profile = source_profile.copy()

    for key in ["blockxsize", "blockysize", "tiled", "interleave", "photometric"]:
        profile.pop(key, None)

    profile.update(
        driver="GTiff",
        width=width,
        height=height,
        count=count,
        dtype=dtype,
        nodata=nodata,
        compress="deflate",
        BIGTIFF="IF_SAFER",
    )

    if width >= 16 and height >= 16:
        profile.update(
            tiled=True,
            blockxsize=min(512, max(16, (width // 16) * 16)),
            blockysize=min(512, max(16, (height // 16) * 16)),
        )
    else:
        profile.update(tiled=False)

    return profile

def convert_sar_band_to_db(band, scale_factor=10000.0):
    band = band.astype(np.float32)
    finite = band[np.isfinite(band)]
    positive = finite[finite > 0]

    if finite.size == 0 or positive.size == 0:
        raise ValueError("S1 band ไม่มีค่าที่ใช้ได้")

    minimum = float(np.min(finite))
    maximum = float(np.max(positive))
    eps = np.float32(1e-8)

    if minimum < -1.0:
        mode = "already_db"
        out = band.copy()
    elif maximum <= 2.0:
        mode = "linear_power"
        out = 10.0 * np.log10(np.maximum(band, eps))
    else:
        mode = f"scaled_linear_power/{scale_factor:g}"
        out = 10.0 * np.log10(np.maximum(band / float(scale_factor), eps))

    invalid = (~np.isfinite(band)) | (band <= 0)
    out[invalid] = -35.0
    out = np.clip(out, -35.0, 5.0).astype(np.float32)

    return out, mode

def prepare_s1_for_terramind(input_path, output_path, scale_factor=10000.0):
    input_path = Path(input_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(input_path) as src:
        raw = src.read().astype(np.float32)
        profile = src.profile.copy()
        descriptions = [str(v).upper() if v is not None else "" for v in src.descriptions]
        width, height = src.width, src.height

    if raw.shape[0] not in (1, 2):
        raise ValueError(f"{input_path.name} มี {raw.shape[0]} bands รองรับเฉพาะ 1 หรือ 2 bands")

    print("\n" + "=" * 100)
    print("PREPARE S1")
    print("=" * 100)
    print("File       :", input_path.name)
    print("Band count :", raw.shape[0])
    print("Shape      :", raw.shape)

    if raw.shape[0] == 1:
        vv_db, mode = convert_sar_band_to_db(raw[0], scale_factor)
        vh_db = np.clip(vv_db + VH_OFFSET_DB, -35.0, 5.0).astype(np.float32)
        vh_db[vv_db <= -35.0] = -35.0
        handling = "single band treated as VV; VH approximated"
        scale_mode = mode
    else:
        medians = []
        for band in raw:
            valid = band[np.isfinite(band) & (band > 0)]
            medians.append(float(np.median(valid)) if valid.size else -np.inf)

        if "VV" in descriptions[0] and "VH" in descriptions[1]:
            vv_idx, vh_idx = 0, 1
        elif "VH" in descriptions[0] and "VV" in descriptions[1]:
            vv_idx, vh_idx = 1, 0
        elif medians[1] > medians[0]:
            vv_idx, vh_idx = 1, 0
        else:
            vv_idx, vh_idx = 0, 1

        vv_db, vv_mode = convert_sar_band_to_db(raw[vv_idx], scale_factor)
        vh_db, vh_mode = convert_sar_band_to_db(raw[vh_idx], scale_factor)
        handling = f"two bands reordered to VV,VH; source indices={vv_idx + 1},{vh_idx + 1}"
        scale_mode = f"VV={vv_mode}; VH={vh_mode}"

    output = np.stack([vv_db, vh_db], axis=0).astype(np.float32)

    profile = make_safe_profile(profile, width, height, 2, "float32", -35.0)

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(output)
        dst.set_band_description(1, "VV_dB")
        dst.set_band_description(2, "VH_dB")
        dst.update_tags(SOURCE_FILE=str(input_path), SCALE_MODE=scale_mode, BAND_HANDLING=handling)

    print("Scale mode :", scale_mode)
    print("Handling   :", handling)
    print("Output     :", output_path)

    return output_path

def align_s2_to_s1(s2_path, s1_reference_path, output_path):
    s2_path = Path(s2_path)
    s1_reference_path = Path(s1_reference_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(s1_reference_path) as ref:
        dst_crs = ref.crs
        dst_transform = ref.transform
        dst_width = ref.width
        dst_height = ref.height
        ref_profile = ref.profile.copy()

    if dst_crs is None:
        raise ValueError("S1 ไม่มี CRS")

    with rasterio.open(s2_path) as src:
        band_count = src.count

        if band_count not in (3, 4, 13):
            raise ValueError(f"{s2_path.name} มี {band_count} bands รองรับ 3, 4, 13")

        src_crs = src.crs
        if src_crs is None:
            if USE_S1_CRS_WHEN_S2_CRS_MISSING:
                print(f"WARNING: {s2_path.name} ไม่มี CRS จึงใช้ CRS ของ S1: {dst_crs}")
                src_crs = dst_crs
            else:
                raise ValueError("S2 ไม่มี CRS")

        aligned = np.empty((13, dst_height, dst_width), dtype=np.float32)
        for i in range(13):
            aligned[i].fill(float(S2_MEAN[i]))

        if band_count == 13:
            mode = "13_band_original"
            mapping = {i: i - 1 for i in range(1, 14)}
        elif band_count == 3:
            mode = "RGB_3band_expanded_to_13"
            mapping = {1: 3, 2: 2, 3: 1} if S2_RGB_ORDER.upper() == "RGB" else {1: 1, 2: 2, 3: 3}
        else:
            mode = "RGBNIR_4band_expanded_to_13"
            mapping = {1: 3, 2: 2, 3: 1, 4: 7} if S2_RGB_ORDER.upper() == "RGB" else {1: 1, 2: 2, 3: 3, 4: 7}

        print("\n" + "=" * 100)
        print("PREPARE S2")
        print("=" * 100)
        print("File         :", s2_path.name)
        print("Source bands :", band_count)
        print("Mode         :", mode)

        for src_band, dst_idx in mapping.items():
            reproject(
                source=rasterio.band(src, src_band),
                destination=aligned[dst_idx],
                src_transform=src.transform,
                src_crs=src_crs,
                src_nodata=src.nodata,
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                dst_nodata=float(S2_MEAN[dst_idx]),
                resampling=Resampling.bilinear,
            )

    profile = make_safe_profile(ref_profile, dst_width, dst_height, 13, "float32", None)
    names = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B10", "B11", "B12"]

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(aligned)
        for i, name in enumerate(names, start=1):
            dst.set_band_description(i, name)
        dst.update_tags(SOURCE_FILE=str(s2_path), SOURCE_MODE=mode, OUTPUT_BAND_COUNT="13")

    print("Output:", output_path)
    return output_path

In [ ]:
# ============================================================
# 10 — ฟังก์ชัน inference, post-process, export
# ============================================================

import gc
import json
import numpy as np
import pandas as pd
import rasterio
import torch
import geopandas as gpd
import matplotlib.pyplot as plt

from rasterio.windows import Window
from rasterio.features import shapes
from scipy import ndimage
from shapely.geometry import shape
from tqdm.auto import tqdm

def generate_positions(full_size, patch_size, stride):
    if full_size <= patch_size:
        return [0]
    positions = list(range(0, full_size - patch_size + 1, stride))
    last = full_size - patch_size
    if positions[-1] != last:
        positions.append(last)
    return positions

def pad_array(array, target_size, fill_value):
    c, h, w = array.shape
    out = np.full((c, target_size, target_size), fill_value, dtype=np.float32)
    out[:, :h, :w] = array
    return out

def extract_logits(result):
    if torch.is_tensor(result):
        return result
    if hasattr(result, "output"):
        return result.output
    if hasattr(result, "logits"):
        return result.logits
    if isinstance(result, dict):
        for key in ["output", "logits", "prediction", "pred"]:
            if key in result:
                return result[key]
    if isinstance(result, (tuple, list)):
        return result[0]
    raise TypeError(f"Cannot extract logits: {type(result)}")

def create_tile_weight(size):
    x = np.linspace(-1.0, 1.0, size, dtype=np.float32)
    w = np.maximum(1.0 - np.abs(x), 0.05)
    return np.outer(w, w).astype(np.float32)

TILE_WEIGHT = create_tile_weight(PATCH_SIZE)

def predict_flood_probability(s2_path, s1_path):
    with rasterio.open(s2_path) as s2_src, rasterio.open(s1_path) as s1_src:
        if s2_src.count != 13:
            raise ValueError("Prepared S2 ต้องมี 13 bands")
        if s1_src.count != 2:
            raise ValueError("Prepared S1 ต้องมี 2 bands")

        h, w = s1_src.height, s1_src.width
        probability_sum = np.zeros((h, w), dtype=np.float32)
        weight_sum = np.zeros((h, w), dtype=np.float32)

        xs = generate_positions(w, PATCH_SIZE, STRIDE)
        ys = generate_positions(h, PATCH_SIZE, STRIDE)

        progress = tqdm(total=len(xs) * len(ys), desc="TerraMind inference")

        for y in ys:
            for x in xs:
                patch_w = min(PATCH_SIZE, w - x)
                patch_h = min(PATCH_SIZE, h - y)
                window = Window(x, y, patch_w, patch_h)

                s2 = s2_src.read(window=window).astype(np.float32)
                s1 = s1_src.read(window=window).astype(np.float32)

                s2 = pad_array(s2, PATCH_SIZE, 0)
                s1 = pad_array(s1, PATCH_SIZE, -35)

                valid = np.any(s2 != 0, axis=0) | np.any(s1 > -34.9, axis=0)

                s2_norm = (s2 - S2_MEAN[:, None, None]) / S2_STD[:, None, None]
                s1_norm = (s1 - S1_MEAN[:, None, None]) / S1_STD[:, None, None]

                model_input = {
                    "S2L1C": torch.from_numpy(s2_norm[None]).to(device, non_blocking=True),
                    "S1GRD": torch.from_numpy(s1_norm[None]).to(device, non_blocking=True),
                }

                with torch.inference_mode():
                    with torch.autocast(device_type="cuda", dtype=torch.float16):
                        result = model(model_input)
                    logits = extract_logits(result)
                    prob = torch.softmax(logits, dim=1)[0, 1].float().cpu().numpy()

                prob[~valid] = 0.0

                weight = TILE_WEIGHT[:patch_h, :patch_w]
                probability_sum[y:y+patch_h, x:x+patch_w] += prob[:patch_h, :patch_w] * weight
                weight_sum[y:y+patch_h, x:x+patch_w] += weight

                del model_input, result, logits, prob
                progress.update(1)

        progress.close()

    return np.clip(probability_sum / np.maximum(weight_sum, 1e-6), 0.0, 1.0).astype(np.float32)

def clean_flood_mask(probability):
    raw = probability >= FLOOD_THRESHOLD
    labeled, _ = ndimage.label(raw)
    sizes = np.bincount(labeled.ravel())
    keep = sizes >= MIN_COMPONENT_PIXELS
    if keep.size > 0:
        keep[0] = False
    clean = keep[labeled]
    clean = ndimage.binary_closing(clean, structure=np.ones((3, 3), dtype=bool))
    clean = ndimage.binary_fill_holes(clean)
    return raw.astype(np.uint8), clean.astype(np.uint8)

def save_single_band_raster(output_path, array, reference_path, dtype, nodata, description):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(reference_path) as ref:
        profile = make_safe_profile(ref.profile, ref.width, ref.height, 1, dtype, nodata)
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(array.astype(dtype), 1)
        dst.set_band_description(1, description)
    return output_path

def export_flood_polygons(clean_mask, reference_path, output_path, scene_id):
    with rasterio.open(reference_path) as ref:
        transform = ref.transform
        source_crs = ref.crs

    if source_crs is None:
        return None, 0.0, 0

    records = []
    for geom, value in shapes(clean_mask, mask=clean_mask == 1, transform=transform):
        if int(value) == 1:
            records.append({"scene": str(scene_id), "class": "probable_flood", "geometry": shape(geom)})

    if not records:
        return None, 0.0, 0

    gdf = gpd.GeoDataFrame(records, geometry="geometry", crs=source_crs)
    area_gdf = gdf.to_crs(AREA_CRS)
    area_gdf["area_m2"] = area_gdf.geometry.area

    keep = area_gdf["area_m2"] >= MIN_POLYGON_AREA_M2
    gdf = gdf.loc[keep.values].copy()
    area_gdf = area_gdf.loc[keep.values].copy()

    if len(gdf) == 0:
        return None, 0.0, 0

    gdf["area_m2"] = area_gdf["area_m2"].values
    gdf["area_km2"] = gdf["area_m2"] / 1_000_000

    output_path = Path(output_path)
    if output_path.exists():
        output_path.unlink()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(output_path, layer="probable_flood", driver="GPKG")

    return output_path, float(gdf["area_km2"].sum()), len(gdf)

def stretch_rgb(rgb):
    rgb = rgb.astype(np.float32)
    out = np.zeros_like(rgb, dtype=np.float32)
    for c in range(3):
        band = rgb[..., c]
        valid = band[np.isfinite(band) & (band > 0)]
        if valid.size == 0:
            continue
        lo, hi = np.percentile(valid, [2, 98])
        out[..., c] = np.clip((band - lo) / (hi - lo + 1e-6), 0, 1)
    return out

def create_scene_preview(scene_id, s2_path, s1_path, probability, clean_mask, flood_area_km2, output_path):
    with rasterio.open(s2_path) as src:
        rgb_data = src.read([4, 3, 2]).astype(np.float32)
    with rasterio.open(s1_path) as src:
        vv = src.read(1).astype(np.float32)

    rgb = stretch_rgb(np.moveaxis(rgb_data, 0, -1))

    fig, axes = plt.subplots(1, 4, figsize=(24, 7))

    axes[0].imshow(rgb)
    axes[0].set_title(f"Scene {scene_id}\nSentinel-2 RGB")
    axes[0].axis("off")

    axes[1].imshow(vv, cmap="gray", vmin=-25, vmax=0)
    axes[1].set_title("Sentinel-1 VV dB")
    axes[1].axis("off")

    im = axes[2].imshow(probability, cmap="viridis", vmin=0, vmax=1)
    axes[2].set_title("Flood probability")
    axes[2].axis("off")

    axes[3].imshow(rgb)
    axes[3].imshow(np.ma.masked_where(clean_mask == 0, clean_mask), cmap="autumn", alpha=0.60, vmin=0, vmax=1)
    axes[3].set_title(f"Probable flood overlay\nArea = {flood_area_km2:.2f} km²")
    axes[3].axis("off")

    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
    plt.tight_layout()

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    return output_path

In [ ]:
# ============================================================
# 11 — รัน prediction ทุก scene
# ============================================================

import gc
import json
import pandas as pd
from IPython.display import display

summary_rows = []

for pair in scene_pairs:
    scene_id = pair["scene"]

    print("\n" + "=" * 100)
    print(f"PROCESSING SCENE: {scene_id}")
    print("=" * 100)

    scene_root = OUTPUT_ROOT / f"scene_{scene_id}"
    processed_root = scene_root / "processed"
    result_root = scene_root / "outputs"
    processed_root.mkdir(parents=True, exist_ok=True)
    result_root.mkdir(parents=True, exist_ok=True)

    s1_ready = processed_root / f"scene_{scene_id}_S1GRD_db.tif"
    s2_ready = processed_root / f"scene_{scene_id}_S2L1C_13band_aligned.tif"

    probability_path = result_root / "01_flood_probability.tif"
    raw_mask_path = result_root / "02_flood_mask_raw.tif"
    clean_mask_path = result_root / "03_flood_mask_clean.tif"
    polygon_path = result_root / "04_probable_flood_polygons.gpkg"
    preview_path = result_root / "05_flood_prediction_preview.png"
    scene_info_path = result_root / "06_scene_metadata.json"

    prepare_s1_for_terramind(pair["s1"], s1_ready, S1_SCALE_FACTOR)
    align_s2_to_s1(pair["s2"], s1_ready, s2_ready)

    probability = predict_flood_probability(s2_ready, s1_ready)
    raw_mask, clean_mask = clean_flood_mask(probability)

    save_single_band_raster(probability_path, probability, s1_ready, "float32", -9999, "Flood probability")
    save_single_band_raster(raw_mask_path, raw_mask, s1_ready, "uint8", 255, "Raw probable flood mask")
    save_single_band_raster(clean_mask_path, clean_mask, s1_ready, "uint8", 255, "Clean probable flood mask")

    saved_polygon_path, flood_area_km2, polygon_count = export_flood_polygons(
        clean_mask=clean_mask,
        reference_path=s1_ready,
        output_path=polygon_path,
        scene_id=scene_id,
    )

    create_scene_preview(
        scene_id=scene_id,
        s2_path=s2_ready,
        s1_path=s1_ready,
        probability=probability,
        clean_mask=clean_mask,
        flood_area_km2=flood_area_km2,
        output_path=preview_path,
    )

    scene_info = {
        "scene": str(scene_id),
        "s1_input": str(pair["s1"]),
        "s2_input": str(pair["s2"]),
        "checkpoint": str(CHECKPOINT),
        "threshold": FLOOD_THRESHOLD,
        "patch_size": PATCH_SIZE,
        "stride": STRIDE,
        "flood_area_km2": flood_area_km2,
        "polygon_count": polygon_count,
        "note": "This is a probable flood extent prototype. Accuracy depends on available S1/S2 bands.",
    }

    scene_info_path.write_text(json.dumps(scene_info, ensure_ascii=False, indent=2), encoding="utf-8")

    summary_rows.append(
        {
            "scene": str(scene_id),
            "source": pair["source"],
            "threshold": FLOOD_THRESHOLD,
            "flood_area_km2": flood_area_km2,
            "flood_pixels": int(clean_mask.sum()),
            "polygon_count": polygon_count,
            "s1_input": str(pair["s1"]),
            "s2_input": str(pair["s2"]),
            "probability_tif": str(probability_path),
            "clean_mask_tif": str(clean_mask_path),
            "polygon_gpkg": str(saved_polygon_path) if saved_polygon_path else "",
            "preview_png": str(preview_path),
        }
    )

    print(f"Scene {scene_id} completed")
    print(f"Flood area: {flood_area_km2:.3f} km²")
    print(f"Preview   : {preview_path}")

    del probability, raw_mask, clean_mask
    gc.collect()
    torch.cuda.empty_cache()

summary_df = pd.DataFrame(summary_rows)

summary_csv = OUTPUT_ROOT / "flood_prediction_summary.csv"
summary_json = OUTPUT_ROOT / "flood_prediction_summary.json"

summary_df.to_csv(summary_csv, index=False)
summary_json.write_text(json.dumps(summary_rows, ensure_ascii=False, indent=2), encoding="utf-8")

print("\n" + "=" * 100)
print("ALL SCENES COMPLETED")
print("=" * 100)
display(summary_df)
print("Summary CSV :", summary_csv)
print("Summary JSON:", summary_json)
print("Output root :", OUTPUT_ROOT)

In [ ]:
# ============================================================
# 12 — แสดง gallery ของ preview ทั้งหมด
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

preview_files = sorted(OUTPUT_ROOT.rglob("05_flood_prediction_preview.png"))

if not preview_files:
    raise FileNotFoundError("ไม่พบ preview PNG กรุณารัน prediction cell ก่อน")

fig, axes = plt.subplots(len(preview_files), 1, figsize=(22, 7 * len(preview_files)))

if len(preview_files) == 1:
    axes = [axes]

for ax, path in zip(axes, preview_files):
    img = mpimg.imread(path)
    ax.imshow(img)
    ax.set_title(str(path.relative_to(OUTPUT_ROOT)))
    ax.axis("off")

plt.tight_layout()
plt.show()


หลังรันเสร็จ ผลลัพธ์อยู่ที่:

```text
Flood_Prediction_Package/outputs/
```

แต่ละ scene จะมีไฟล์:

```text
01_flood_probability.tif       ค่า probability 0–1
02_flood_mask_raw.tif          mask จาก threshold ตรง ๆ
03_flood_mask_clean.tif        mask หลัง post-processing
04_probable_flood_polygons.gpkg polygon เปิดใน QGIS/ArcGIS
05_flood_prediction_preview.png ภาพสรุปสำหรับรายงาน
06_scene_metadata.json          metadata ของ scene
```

และมีไฟล์รวม:

```text
flood_prediction_summary.csv
flood_prediction_summary.json
```


ผลลัพธ์นี้เป็น workflow ต้นแบบสำหรับ flood detection โดยใช้ TerraMind Base ที่ fine-tune จาก Sen1Floods11 และนำมาประยุกต์กับข้อมูลพื้นที่ศึกษา ผลลัพธ์ควรถูกตีความเป็น probable flood / water extent โดยเฉพาะกรณีที่ข้อมูล Sentinel-1 หรือ Sentinel-2 ไม่ครบทุก band ตามชุดฝึก

## ถ้าต้องการผลที่ดีขึ้น

1. ใช้ Sentinel-1 แบบ VV+VH จริง
2. ใช้ Sentinel-2 L1C 13 bands จริง
3. ใช้ภาพก่อนน้ำท่วมและหลังน้ำท่วม
4. ตัด permanent water ออก
5. เพิ่ม DEM / slope mask
6. ทำ manual label บางพื้นที่เพื่อปรับ threshold